<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install gradio

In [ ]:

import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

financial_evidence = [
    {
        "id": "E1",
        "topic": "Credit Score",
        "source": "Financial Education",
        "text": "Paying bills on time and maintaining a good payment history can help improve a credit score."
    },
    {
        "id": "E2",
        "topic": "Credit Utilization",
        "source": "Financial Education",
        "text": "Credit utilization is the percentage of available revolving credit that a person is using. Lower utilization is generally better for credit scoring."
    },
    {
        "id": "E3",
        "topic": "Loan Approval",
        "source": "Financial Education",
        "text": "Loan approval may depend on credit history, income, existing debts, repayment capacity, and the lender's eligibility criteria. A high credit score does not guarantee approval."
    },
    {
        "id": "E4",
        "topic": "Debt-to-Income Ratio",
        "source": "Financial Education",
        "text": "Debt-to-income ratio compares monthly debt payments with gross monthly income. It is calculated by dividing monthly debt payments by gross monthly income and multiplying by 100."
    },
    {
        "id": "E5",
        "topic": "Budgeting",
        "source": "Financial Education",
        "text": "A budget tracks income and expenses. An emergency fund can help cover unexpected expenses without relying entirely on borrowing."
    },
    {
        "id": "E6",
        "topic": "Credit Report",
        "source": "Financial Education",
        "text": "Reviewing a credit report regularly can help identify errors. Disputed errors should be reported through the relevant credit bureau's official process."
    }
]

evidence_texts = [item["text"] for item in financial_evidence]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

evidence_matrix = vectorizer.fit_transform(evidence_texts)



def financial_assistant(question):
    if not question or not question.strip():
        return "Please enter a financial question."

    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(
        question_vector, evidence_matrix
    )[0]

    ranked_indices = scores.argsort()[::-1]
    matches = []

    for index in ranked_indices:
        if scores[index] <= 0:
            continue

        matches.append({
            "item": financial_evidence[index],
            "score": float(scores[index])
        })

        if len(matches) == 3:
            break

    if not matches:
        return (
            "### Insufficient Evidence\n"
            "I couldn't find relevant information in the current "
            "knowledge base. Please try another question or add "
            "relevant financial documents."
        )

    answer = "## Financial Explanation\n\n"
    answer += (
        "This response is based on the matching information "
        "in the current knowledge base.\n\n"
    )

    for match in matches:
        item = match["item"]
        answer += f"**{item['topic']}**\n\n"
        answer += f"{item['text']}\n\n"
        answer += f"Evidence ID: `{item['id']}`\n\n"

    answer += "---\n## Supporting Sources\n\n"

    for match in matches:
        item = match["item"]
        answer += (
            f"- **{item['id']} — {item['topic']}** "
            f"({item['source']})\n"
        )

    answer += (
        "\n\n**Note:** This is general financial education, "
        "not personalized financial advice. Verify important "
        "decisions with reliable official sources or a qualified "
        "financial professional."
    )

    return answer



with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 💰 Financial RAG Assistant

        Ask questions about credit scores, loans, budgeting,
        credit utilization, and debt-to-income ratio.

        Answers are matched against the available knowledge base.
        """
    )

    question = gr.Textbox(
        label="Enter your financial question",
        placeholder="Example: How can I improve my credit score?",
        lines=3
    )

    submit_btn = gr.Button("Get Answer", variant="primary")
    clear_btn = gr.Button("Clear")

    answer = gr.Markdown(label="Assistant Response")

    submit_btn.click(
        fn=financial_assistant,
        inputs=question,
        outputs=answer
    )

    question.submit(
        fn=financial_assistant,
        inputs=question,
        outputs=answer
    )

    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[question, answer]
    )

    gr.Examples(
        examples=[
            ["How can I improve my credit score?"],
            ["What is credit utilization?"],
            ["Does a high credit score guarantee loan approval?"],
            ["How do I calculate my debt-to-income ratio?"],
            ["How can I plan for unexpected expenses?"]
        ],
        inputs=question
    )


demo.launch(share=True, debug=True)


/tmp/ipykernel_7085/733996520.py:122: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea6d7548bf88cc7db7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
